In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os

In [ ]:
train_data = pd.read_csv(os.path.join('data', 'train-data.tsv'), sep='\t', header=None, names=['label', 'message'])
test_data = pd.read_csv(os.path.join('data', 'valid-data.tsv'), sep='\t', header=None, names=['label', 'message'])

train_data['label'] = train_data['label'].map({'ham': 0, 'spam': 1})
test_data['label'] = test_data['label'].map({'ham': 0, 'spam': 1})

train_texts = train_data['message'].values
train_labels = train_data['label'].values
test_texts = test_data['message'].values
test_labels = test_data['label'].values

In [ ]:
vocab_size = 10000
max_length = 200
embedding_dim = 16
trunc_type = 'post'
padding_type = 'post'
oov_tok = '<OOV>'

tokenizer = tf.keras.preprocessing.text.Tokenizer(
    num_words=vocab_size, oov_token=oov_tok
)
tokenizer.fit_on_texts(train_texts)

train_sequences = tokenizer.texts_to_sequences(train_texts)
train_padded = tf.keras.preprocessing.sequence.pad_sequences(
    train_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type
)

test_sequences = tokenizer.texts_to_sequences(test_texts)
test_padded = tf.keras.preprocessing.sequence.pad_sequences(
    test_sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type
)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(24, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    train_padded, train_labels,
    epochs=30,
    validation_data=(test_padded, test_labels),
    verbose=1
)

In [ ]:
def predict_message(msg):
    seq = tokenizer.texts_to_sequences([msg])
    padded = tf.keras.preprocessing.sequence.pad_sequences(
        seq, maxlen=max_length, padding=padding_type, truncating=trunc_type
    )
    prob = float(model.predict(padded, verbose=0)[0][0])
    label = 'spam' if prob > 0.5 else 'ham'
    return [prob, label]

In [ ]:
print(predict_message("You've won a free ticket! Claim now"))
print(predict_message("Hey, are we still meeting at 5?"))

In [ ]:
test_loss, test_acc = model.evaluate(test_padded, test_labels, verbose=2)
print(f'Test accuracy: {test_acc:.4f}')